## **Cargar CSV Pandas** 🐍
Es una biblioteca de Python de código abierto especializada en la manipulación y el análisis de datos. Su nombre proviene del término en inglés "Panel Data" (datos de panel).

Estructuras de datos principalesSeries: 
* Una matriz unidimensional con etiquetas que puede contener cualquier tipo de datos.
* DataFrame: Una tabla bidimensional con filas y columnas etiquetadas, similar a una hoja de cálculo de Excel o una tabla SQL.

Para qué sirve
* Leer y escribir archivos: Permite importar y exportar datos en formatos como CSV, Excel, JSON y bases de datos SQL.
* Limpieza de datos: Facilita la detección, eliminación o relleno de valores faltantes o nulos.
* Transformación y análisis: Permite filtrar, agrupar, ordenar y combinar grandes volúmenes de información de manera rápida y eficiente.

In [27]:
#pip install pandas
#pip install numpy

import pandas as pd

#importar archivo csv
df = pd.read_csv('dataset_ventas.csv')

# df.head()# 5 primeras filas
# df.tail()# 5 últimas filas
# df.sample(n=5) # 5 filas aleatorias
# df.info() # información general sobre el DataFrame
# df.describe() # información sobre el DataFrame

df.shape # número de filas y columnas

(1000, 7)

### 1. Manejo de Valores faltantes y duplicados

In [ ]:
#1. Identificar valores nulos en el DataFrame
df.isnull().sum()

#2. Eliminar filas con valores nulos
#df.dropna()

# 3. Rellenar valores nulos con un valor específico
#df.fillna(0)

# 4. Rellenar valores nulos con la media de la columna
#df['cantidad'] = df['cantidad'].fillna(df['cantidad'].mean())

# 5. Verificar si hay valores duplicados
df.duplicated().sum()

# 6. Eliminar valores duplicados
#df.drop_duplicates()

np.int64(0)

### 2. Codifique variables categóricas usando LabelEncoder y One-Hot Encoding.

In [ ]:
from sklearn.preprocessing import LabelEncoder

# ============ 1) LABEL ENCODER ============
# Convierte cada categoría única en un número entero (0, 1, 2, ...)
le_ciudad = LabelEncoder()
df['ciudad_label'] = le_ciudad.fit_transform(df['ciudad'])

le_categoria = LabelEncoder()
df['categoria_label'] = le_categoria.fit_transform(df['categoria'])

print('LabelEncoder aplicado:')
print(df[['ciudad', 'ciudad_label', 'categoria', 'categoria_label']].head())

# ============ 2) ONE-HOT ENCODING ============
# Crea una columna binaria (0/1) por cada categoría.
# Guardamos el objetivo ANTES de codificar (lo usaremos con SMOTE)
y = df['categoria'].copy()

df = pd.get_dummies(df, columns=['ciudad', 'categoria', 'producto'])

print()
print('Columnas después de One-Hot Encoding:')
print(list(df.columns))

### 3. Aplique normalización y estandarización.

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Columnas numéricas a transformar
numeric_cols = ['cantidad', 'precio_unitario', 'total_venta']

# ============ 1) NORMALIZACIÓN (Min-Max) ============
# Escala los valores al rango [0, 1]
scaler_minmax = MinMaxScaler()
df_norm = df.copy()
df_norm[numeric_cols] = scaler_minmax.fit_transform(df[numeric_cols])

print('Después de NORMALIZAR (Min-Max) -> rango [0, 1]:')
print(df_norm[numeric_cols].describe().round(2))

# ============ 2) ESTANDARIZACIÓN (Z-score) ============
# Re-escala con media = 0 y desviación estándar = 1
scaler_std = StandardScaler()
df_std = df.copy()
df_std[numeric_cols] = scaler_std.fit_transform(df[numeric_cols])

print()
print('Después de ESTANDARIZAR (Z-score) -> media 0, desv. estándar 1:')
print(df_std[numeric_cols].describe().round(2))

### 4. Balancee la variable objetivo usando SMOTE.

In [ ]:
#pip install imbalanced-learn
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

# Características: columnas numéricas ya codificadas (sin la fecha)
X = df.drop(columns=['fecha'])

# Variable objetivo: guardada antes de la codificación
print('Distribución original de la variable objetivo:')
print(y.value_counts())

# División train/test (estratificada para conservar las proporciones)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print()
print('Distribución ANTES de SMOTE (solo train):')
print(y_train.value_counts())

# SMOTE genera muestras sintéticas de las clases minoritarias
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_train, y_train)

print()
print('Distribución DESPUÉS de SMOTE (balanceada):')
print(y_res.value_counts())

### 5. Muestre el conjunto de datos final listo para entrenar un modelo de clasificación.

In [ ]:
# Convertir el resultado de SMOTE a DataFrame (por si viene como array)
X_res = pd.DataFrame(X_res, columns=X_train.columns)

# Conjunto final balanceado y listo para entrenar
df_final = X_res.copy()
df_final['categoria'] = y_res

print(f'✅ Dataset final: {df_final.shape[0]} filas y {df_final.shape[1]} columnas')
print('📊 Clases balanceadas:')
print(df_final['categoria'].value_counts())

print()
print('Primeras filas del dataset final:')
df_final.head()